In [ ]:
import redis
import base64

# Conexão com o Redis
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

def salvar_imagem(caminho_imagem, chave_redis):
    with open(caminho_imagem, "rb") as image_file:
        # 1. Converte o binário para Base64
        bn_data = base64.b64encode(image_file.read())
        # 2. Converte bytes para string para salvar no Redis
        base64_string = bn_data.decode('utf-8')
        
        # 3. Salva no Redis (pode ser em um campo de um Hash ou String simples)
        r.set(chave_redis, base64_string)
        print(f"Imagem salva com sucesso na chave: {chave_redis}")

def recuperar_imagem(chave_redis):
    # Recupera a string do Redis
    img_base64 = r.get(chave_redis)
    return img_base64

# Exemplo de uso:
salvar_imagem("sua_foto.jpg", "usuario:10:foto")

In [2]:
from flask import Flask, jsonify
import redis
import base64

app = Flask(__name__)

# Conexão com o Redis (ajuste host/porta se necessário)
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

@app.route('/get-image/<chave>', methods=['GET'])
def obter_imagem(chave):
    # 1. Busca a string Base64 que salvamos anteriormente
    img_base64 = r.get(chave)
    
    if not img_base64:
        return jsonify({"erro": "Imagem não encontrada"}), 404
    
    # 2. Retorna o JSON com a string para o Front-end
    return jsonify({
        "chave": chave,
        "base64_data": img_base64
    })

if __name__ == '__main__':
    app.run(debug=True, port=5000)

ModuleNotFoundError: No module named 'flask'